In [ ]:
import duckdb

# 1. Criar (ou conectar) a uma base de dados local dentro da pasta 'data'
conexao = duckdb.connect('data/logistica.db')

# 2. Ingerir os CSVs limpos diretamente para tabelas SQL de alta performance
conexao.execute("""
    CREATE TABLE IF NOT EXISTS f_movimentacao AS
    SELECT * FROM read_csv_auto('data/f_Movimentacao_Logistica.csv');
""")

conexao.execute("""
    CREATE TABLE IF NOT EXISTS d_clientes AS
    SELECT * FROM read_csv_auto('data/d_Clientes.csv');
""")

# 3. Teste de Fogo: Fazer um JOIN em SQL puro para encontrar os gargalos geográficos
query_analise = """
    SELECT
        c.customer_state as UF_Destino,
        COUNT(f.order_id) as total_entregas_atrasadas,
        ROUND(AVG(f.dias_variacao_sla), 1) as media_dias_atraso
    FROM f_movimentacao f
    JOIN d_clientes c ON f.customer_id = c.customer_id
    WHERE f.status_entrega = 'Atrasado'
    GROUP BY c.customer_state
    ORDER BY total_entregas_atrasadas DESC
    LIMIT 5;
"""

print("--- TOP 5 Estados com Pior Performance (SQL) ---")
# O comando .df() converte o resultado do SQL de volta para uma tabela visual agradável
display(conexao.execute(query_analise).df())

# 4. Fechar a ligação para proteger o ficheiro da base de dados
conexao.close()
print("\nBase de dados 'logistica.db' criada e pronta para o Power BI!")

--- TOP 5 Estados com Pior Performance (SQL) ---


,UF_Destino,total_entregas_atrasadas,media_dias_atraso
0,SP,1820,8.3
1,RJ,1495,13.5
2,MG,519,8.4
3,BA,396,12.0
4,RS,325,10.2



Base de dados 'logistica.db' criada e pronta para o Power BI!
